# ENV and Loading model

In [5]:
import sys, os
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'

from core.model.spatial_llava import load_model
model, processor = load_model(use_lora=True, device='cuda')
print("Model loaded, ready to train.")

[load_model] use_lora=True  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:06<00:00, 104.28it/s]


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
Model loaded, ready to train.


# Run the main model with LoRa and created MLP

In [1]:
import sys, os, argparse, importlib, torch, gc
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from core.model.spatial_llava import load_model
import pipeline.step2_train_main as s2
importlib.reload(s2)

# ── 清理 ─────────────────────────────────────────────────
try:
    del model, processor
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM before: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ── 加载模型 ─────────────────────────────────────────────
model, processor = load_model(use_lora=True, device='cuda')
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ── 正式训练配置 ─────────────────────────────────────────
args = argparse.Namespace(
    lora_rank=16,
    epochs=10,
    batch_size=8,
    lr=2e-4,
    weight_decay=0.0,
    max_length=600,
    max_samples=20000,
    val_max_samples=1000,
    num_workers=2,
    log_every=200,
    n_examples=20,
    seed=42,
    resume=False,
)

print("\n" + "="*60)
print("  🚀 Step 2: Formal Training")
print(f"  samples : 20,000 train | 1,000 val")
print(f"  epochs  : {args.epochs}")
print(f"  batch   : {args.batch_size}")
print(f"  lr      : {args.lr}")
print(f"  ETA     : ~7 hours")
print("="*60 + "\n")

s2.main(args, model=model, processor=processor)
print(f"\nVRAM peak: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VRAM before: 0.0 GB
[load_model] use_lora=True  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:09<00:00, 72.22it/s] 


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
VRAM after load: 14.2 GB

  🚀 Step 2: Formal Training
  samples : 20,000 train | 1,000 val
  epochs  : 10
  batch   : 8
  lr      : 0.0002
  ETA     : ~7 hours

[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step2_20260410_1531.log
[15:31:10] ============================================================
[15:31:10]   Step 2: Train SpatialLLaVA (LLaVA + LoRA + Head)
[15:31:10]   Config: {
  "epochs": 10,
  "batch_size": 8,
  "lr": 0.0002,
  "lora_rank": 16,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": true,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[15:31:10] ============================================================
[15:31:10] [1/5] Loading model ...
[15:31:10]   Using pre-loaded model, skipping load.
[15:31:10] [2/5] Loading data ...
[RefC

# Run ablation model

In [1]:
import sys, os, argparse, importlib, torch, gc
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from core.model.spatial_llava import load_model
import pipeline.step3_train_ablation as s3
importlib.reload(s3)

try:
    del model, processor
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model, processor = load_model(use_lora=False, device='cuda')

args = argparse.Namespace(
    epochs=10,
    batch_size=8,
    lr=2e-4,
    weight_decay=0.0,
    max_length=600,
    num_workers=2,
    log_every=200,
    n_examples=20,
    seed=42,
    max_samples=20000,
    val_max_samples=1000,
    resume=False,
    lora_rank=16,
)

print("\n" + "="*60)
print("  🔬 Step 3: Ablation Training")
print(f"  samples : 20,000 train | 1,000 val")
print(f"  epochs  : {args.epochs}")
print(f"  LoRA    : ❌ disabled")
print(f"  ETA     : ~3.5 hours")  # 2.3it/s，比 step2 快一倍
print("="*60 + "\n")

s3.main(args, model=model, processor=processor)
print(f"\nVRAM peak: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[load_model] use_lora=False  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:09<00:00, 74.47it/s] 


[SpatialLLaVA] Freezing entire backbone (ablation mode) ...
[SpatialLLaVA] Head params: 2,230,020

  🔬 Step 3: Ablation Training
  samples : 20,000 train | 1,000 val
  epochs  : 10
  LoRA    : ❌ disabled
  ETA     : ~3.5 hours

[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step3_20260411_0137.log
[01:37:50] ============================================================
[01:37:50]   Step 3: Ablation — frozen backbone + head only
[01:37:50]   Config: {
  "epochs": 10,
  "batch_size": 8,
  "lr": 0.0002,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": false,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[01:37:50] ============================================================
[01:37:50] [1/5] Loading model ...
[01:37:50]   Using pre-loaded model, skipping load.
[01:37:50]   Trainable params: 2,230,020 (should be ~2.2M head only)
[01:37:50] [2/5] Loading data ...
[RefCOCODataset] train: 20,000 samples
[RefCOCODataset] val: 1,000 samples
[make_loaders] train=20,000  val=1,000  ba

Epoch 1:   8%|▊         | 200/2500 [01:24<15:52,  2.41it/s]

...

Epoch 10:  96%|█████████▌| 2400/2500 [16:49<00:42,  2.38it/s]

[04:40:15]   Epoch 10 | step 2400/2500 | loss=0.9962 | 1009.0s
    Epoch   [███████████████████░]  96.0%  ETA: 0.7min
    Overall [███████████████████░]  99.6%  ETA: 0.7min


Epoch 10: 100%|██████████| 2500/2500 [17:31<00:00,  2.38it/s]


[04:40:57]   [Train] loss=0.9580  IoU=0.2898


[04:41:50]   [Val] epoch=10  loss=1.0000  IoU=0.2672  RMSE=0.2308  MAE=0.1879
[04:41:50]   LR → 1.00e-06
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/ablation/latest.pth  (26.8 MB)
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/ablation/best.pth  (26.8 MB)
  [ckpt] ★ New best IoU: 0.2672
[04:41:50] 
[5/5] Saving results ...
[04:41:50]   Saved metrics → /tmp/SPATIAL-LLAVA/results/ablation/metrics.json
[04:41:50]   Saved predictions → /tmp/SPATIAL-LLAVA/results/ablation/predictions.json
[04:41:52]   Saved 20 example PNGs → /tmp/SPATIAL-LLAVA/results/ablation/examples
[04:41:52] 
✅ Step 3 complete!
[04:41:52]   Best val IoU  : 0.2672
[04:41:52]   Checkpoint    : /tmp/SPATIAL-LLAVA/checkpoints/ablation/best.pth
[04:41:52]   Results       : /tmp/SPATIAL-LLAVA/results/ablation

VRAM peak: 18.4 GB


# Evaluation

In [1]:
import sys, os, importlib, torch, gc
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import pipeline.step4_evaluate as s4
importlib.reload(s4)
ckpt2 = torch.load('/tmp/SPATIAL-LLAVA/checkpoints/main/best.pth', map_location='cuda')
print(list(ckpt2["model_state"].keys())[:20])

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_112972/401840723.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` f

['backbone.model.language_model.base_model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'backbone.model.language_model.base_model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'backbone.model.language_model.base_model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'backbone.model.language_model.base_model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'backbone.model.language_model.base_model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'backbone.model.language_model.base_model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'backbone.model.language_model.base_model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'backbone.model.language_model.base_model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'backbone.model.language_model.base_model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'backbone.model.language_model.base_model.model.layers.1.self_attn.k_proj.lora_B.default.weight', 'backbone.model.lan

In [2]:
import sys, os, importlib, torch, gc
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import pipeline.step4_evaluate as s4
importlib.reload(s4)

gc.collect()
torch.cuda.empty_cache()

import argparse
args = argparse.Namespace(batch_size=8)
s4.main(args)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step4_20260411_0734.log
[07:34:54] ============================================================
[07:34:54]   Step 4: Final Evaluation — All Three Models
[07:34:54]   Device: cuda
[07:34:54] ============================================================
[07:34:54] 
[07:34:54]   [1/3] Baseline: Standard LLaVA + regex
[07:34:54] ============================================================
  [StandardLLaVA] Loading processor: llava-hf/llava-1.5-7b-hf ...
  [StandardLLaVA] Loading model weights (~14GB)...
  [StandardLLaVA] Cache: /tmp/SPATIAL-LLAVA/weights


Loading weights: 100%|██████████| 686/686 [00:00<00:00, 7114.19it/s]


  [StandardLLaVA] ✓ Loaded (7.1B params, all frozen)
  [RefCOCODatasetPIL] Loading test: /tmp/SPATIAL-LLAVA/data/refcoco_test.pkl
  [RefCOCODatasetPIL] ✓ Loaded 1,975 samples (v3 format)
[07:35:02]   Test samples: 1,975
[07:35:54]   [200/1975] parse=100.0% elapsed=0.9min
[07:36:42]   [400/1975] parse=100.0% elapsed=1.7min
[07:37:26]   [600/1975] parse=100.0% elapsed=2.4min
[07:38:10]   [800/1975] parse=100.0% elapsed=3.1min
[07:38:52]   [1000/1975] parse=100.0% elapsed=3.8min
[07:39:34]   [1200/1975] parse=100.0% elapsed=4.5min
[07:40:17]   [1400/1975] parse=100.0% elapsed=5.2min
[07:40:58]   [1600/1975] parse=100.0% elapsed=5.9min
[07:41:39]   [1800/1975] parse=100.0% elapsed=6.6min
[07:42:18]   [1975/1975] parse=100.0% elapsed=7.3min
[07:42:18]   IoU=0.0971  parse=100.0%
[07:42:18] 
[07:42:18]   Evaluating: Ablation (Frozen + Head)
[07:42:18] ============================================================
[load_model] use_lora=False  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5

Loading weights: 100%|██████████| 686/686 [00:11<00:00, 60.88it/s] 


[SpatialLLaVA] Freezing entire backbone (ablation mode) ...
[SpatialLLaVA] Head params: 2,230,020
[07:42:38]   Loading: /tmp/SPATIAL-LLAVA/checkpoints/ablation/best.pth
[07:42:38]   Loaded epoch=9
[RefCOCODataset] test: 1,975 samples
[07:42:38]   Test samples: 1,975


/tmp/SPATIAL-LLAVA/pipeline/step4_evaluate.py:148: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[07:43:00]   [400/1975] (20.3%) elapsed=0.4min
[07:43:21]   [800/1975] (40.5%) elapsed=0.7min
[07:43:41]   [1200/1975] (60.8%) elapsed=1.1min
[07:44:02]   [1600/1975] (81.0%) elapsed=1.4min
[07:44:22]   [1975/1975] (100.0%) elapsed=1.7min
[07:44:22]   IoU=0.2843  RMSE=0.2238  MAE=0.1769
[07:44:22] 
[07:44:22]   Evaluating: Main (LoRA + Head)
[07:44:22] ============================================================
[load_model] use_lora=True  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:05<00:00, 119.43it/s]


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
[07:44:47]   Loading: /tmp/SPATIAL-LLAVA/checkpoints/main/best.pth
[07:44:47]   Loaded epoch=6
[RefCOCODataset] test: 1,975 samples
[07:44:47]   Test samples: 1,975
[07:45:09]   [400/1975] (20.3%) elapsed=0.4min
[07:45:31]   [800/1975] (40.5%) elapsed=0.7min
[07:45:52]   [1200/1975] (60.8%) elapsed=1.1min
[07:46:14]   [1600/1975] (81.0%) elapsed=1.4min
[07:46:34]   [1975/1975] (100.0%) elapsed=1.8min
[07:46:34]   IoU=0.3861  RMSE=0.1719  MAE=0.1194
[07:46:34] 
[07:46:34]   FINAL RESULTS (RefCOCO Test Split)
[07:46:34] ============================================================
[07:46:34]   Baseline  IoU: 0.0971
[07:46:34]   Ablation  IoU: 0.2843  (+192.7%)
[07:46:34]   Main      IoU: 0.3861  (+297.5%)
[07:46:34]   Saved → /tmp/SPATIAL-LLAVA/results/evaluation
[

# Training speed test and optimization

from pipeline.profile_step import run
run(model, processor, batch_size=4, n_steps=10, num_workers=2)

# Speed optimization with Flash-Attn and test different settings

In [2]:
import importlib
import torch
import pipeline.profile_step as ps
importlib.reload(ps)

print("="*62)
print("Test 1: batch_size=8, num_workers=2")
print("="*62)
ps.run(model, processor, batch_size=8, n_steps=10, num_workers=2)

torch.cuda.empty_cache()

print("\n"+"="*62)
print("Test 2: batch_size=16, num_workers=2")
print("="*62)
ps.run(model, processor, batch_size=16, n_steps=10, num_workers=2)

torch.cuda.empty_cache()

print("\n"+"="*62)
print("Test 3: batch_size=4, num_workers=4")
print("="*62)
ps.run(model, processor, batch_size=4, n_steps=10, num_workers=4)

torch.cuda.empty_cache()

Test 1: batch_size=8, num_workers=2
[RefCOCODataset] train: 160 samples
[RefCOCODataset] val: 3,811 samples
[make_loaders] train=160  val=3,811  batch=8  workers=2

Profiling 10 steps (batch_size=8)...

step     data    forward     loss   backward    optim    total
--------------------------------------------------------------
   0     223ms      1786ms      50ms      1030ms      60ms    3152ms
   1       0ms       432ms       3ms       528ms     162ms    1127ms
   2       0ms       429ms       3ms       525ms       4ms     962ms
   3       0ms       431ms       4ms       526ms       4ms     967ms
   4       0ms       431ms       3ms       527ms       4ms     966ms
   5       0ms       431ms       3ms       527ms       4ms     966ms
   6       0ms       431ms       3ms       527ms       4ms     966ms
   7       0ms       435ms       3ms       530ms       4ms     973ms
   8       0ms       432ms       2ms       528ms       4ms     968ms
   9       0ms       432ms       3ms       528ms  

# Test the different parameter combos

In [1]:
import sys, os, argparse, importlib, torch
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from core.model.spatial_llava import load_model
import pipeline.step2_train_main as s2
importlib.reload(s2)

for lr, label in [(2e-4, "2e-4"), (5e-5, "5e-5"), (2e-5, "2e-5")]:
    print(f"\n{'='*50}\nTesting lr={label}\n{'='*50}")
    try:
        del model
    except NameError:
        pass
    torch.cuda.empty_cache()
    model, processor = load_model(use_lora=True, device='cuda')
    args = argparse.Namespace(
        epochs=5, batch_size=16,  # ← 8 改成 16
        lr=lr,
        weight_decay=0.0, max_length=600,
        num_workers=2, log_every=50,
        n_examples=0, seed=42,
        max_samples=2000, val_max_samples=300,
        resume=False,
        lora_rank=16,
    )
    s2.main(args, model=model, processor=processor)
    print(f"\nGPU after lr={label}: {torch.cuda.memory_allocated()/1e9:.1f}GB")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Testing lr=2e-4
[load_model] use_lora=True  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:09<00:00, 76.20it/s] 


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step2_20260410_1204.log
[12:04:01] ============================================================
[12:04:01]   Step 2: Train SpatialLLaVA (LLaVA + LoRA + Head)
[12:04:01]   Config: {
  "epochs": 5,
  "batch_size": 16,
  "lr": 0.0002,
  "lora_rank": 16,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": true,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[12:04:01] ============================================================
[12:04:01] [1/5] Loading model ...
[12:04:01]   Using pre-loaded model, skipping load.
[12:04:01] [2/5] Loading data ...
[RefCOCODataset] train: 2,000 samples
[RefCOCODataset] val: 300 samples
[make_loaders] train=2,000  val=300  batch=16  workers=2
[12:04:05] [3/5] Setting up optimizer

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


[12:06:15]   Epoch 1 | step 50/125 | loss=1.2054 | 129.8s
    Epoch   [████████░░░░░░░░░░░░]  40.0%  ETA: 3.2min
    Overall [█░░░░░░░░░░░░░░░░░░░]   8.0%  ETA: 24.9min
[12:08:20]   Epoch 1 | step 100/125 | loss=1.0184 | 255.3s
    Epoch   [████████████████░░░░]  80.0%  ETA: 1.1min
    Overall [███░░░░░░░░░░░░░░░░░]  16.0%  ETA: 22.3min
[12:09:23]   [Train] loss=1.2353  IoU=0.1752
[12:09:40]   [Val] epoch=1  loss=1.0444  IoU=0.2451  RMSE=0.2480  MAE=0.2068
[12:09:40]   LR → 1.81e-04
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/main/latest.pth  (102.5 MB)
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/main/best.pth  (102.5 MB)
  [ckpt] ★ New best IoU: 0.2451
[12:09:40] 
[12:09:40]   Epoch 2/5
[12:09:40] ========================================
[12:11:47]   Epoch 2 | step 50/125 | loss=1.0492 | 127.3s
    Epoch   [████████░░░░░░░░░░░░]  40.0%  ETA: 3.2min
    Overall [█████░░░░░░░░░░░░░░░]  28.0%  ETA: 19.1min
[12:13:54]   Epoch 2 | step 100/125 | loss=1.0065 | 254.2s
    Epoch   [██

Loading weights: 100%|██████████| 686/686 [00:05<00:00, 120.08it/s]


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step2_20260410_1232.log
[12:32:31] ============================================================
[12:32:31]   Step 2: Train SpatialLLaVA (LLaVA + LoRA + Head)
[12:32:31]   Config: {
  "epochs": 5,
  "batch_size": 16,
  "lr": 5e-05,
  "lora_rank": 16,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": true,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[12:32:31] ============================================================
[12:32:31] [1/5] Loading model ...
[12:32:31]   Using pre-loaded model, skipping load.
[12:32:31] [2/5] Loading data ...
[RefCOCODataset] train: 2,000 samples
[RefCOCODataset] val: 300 samples
[make_loaders] train=2,000  val=300  batch=16  workers=2
[12:32:34] [3/5] Setting up optimizer 

Loading weights: 100%|██████████| 686/686 [00:05<00:00, 119.90it/s]


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step2_20260410_1301.log
[13:01:03] ============================================================
[13:01:03]   Step 2: Train SpatialLLaVA (LLaVA + LoRA + Head)
[13:01:03]   Config: {
  "epochs": 5,
  "batch_size": 16,
  "lr": 2e-05,
  "lora_rank": 16,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": true,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[13:01:03] ============================================================
[13:01:03] [1/5] Loading model ...
[13:01:03]   Using pre-loaded model, skipping load.
[13:01:03] [2/5] Loading data ...
[RefCOCODataset] train: 2,000 samples
[RefCOCODataset] val: 300 samples
[make_loaders] train=2,000  val=300  batch=16  workers=2
[13:01:06] [3/5] Setting up optimizer 

# VRAM optimization

In [2]:
import sys, os, argparse, importlib, torch, gc
sys.path.insert(0, '/tmp/SPATIAL-LLAVA')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from core.model.spatial_llava import load_model
import pipeline.step2_train_main as s2
importlib.reload(s2)

# ── 清理旧 run 残留 ──────────────────────────
try:
    del model
    del processor
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# ── 加载模型 ─────────────────────────────────
model, processor = load_model(use_lora=True, device='cuda')

args = argparse.Namespace(
    epochs=1, batch_size=8,
    lr=2e-4,
    weight_decay=0.0, max_length=600,
    num_workers=2, log_every=10,
    n_examples=0, seed=42,
    max_samples=100, val_max_samples=50,
    resume=False, lora_rank=16,
)
s2.main(args, model=model, processor=processor)

[load_model] use_lora=True  device=cuda
[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:04<00:00, 146.29it/s]


[SpatialLLaVA] Applying LoRA to language model ...
[LoRA] LLaVA backbone (LoRA)
  Total params     : 6,620,188,672
  Trainable params : 12,582,912 (0.19%)
  Frozen params    : 6,607,605,760
[SpatialLLaVA] Head params: 2,230,020
[Logger] Writing to /tmp/SPATIAL-LLAVA/logs/step2_20260410_1518.log
[15:18:04] ============================================================
[15:18:04]   Step 2: Train SpatialLLaVA (LLaVA + LoRA + Head)
[15:18:04]   Config: {
  "epochs": 1,
  "batch_size": 8,
  "lr": 0.0002,
  "lora_rank": 16,
  "weight_decay": 0.0,
  "max_length": 600,
  "seed": 42,
  "use_lora": true,
  "model": "llava-hf/llava-1.5-7b-hf"
}
[15:18:04] ============================================================
[15:18:04] [1/5] Loading model ...
[15:18:04]   Using pre-loaded model, skipping load.
[15:18:04] [2/5] Loading data ...
[RefCOCODataset] train: 100 samples
[RefCOCODataset] val: 50 samples
[make_loaders] train=100  val=50  batch=8  workers=2
[15:18:08] [3/5] Setting up optimizer ...
[15

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


[15:18:23]   Epoch 1 | step 10/12 | loss=1.4391 | 15.5s
    Epoch   [████████████████░░░░]  83.3%  ETA: 0.1min
    Overall [████████████████░░░░]  83.3%  ETA: 0.1min
[15:18:26]   [Train] loss=1.5773  IoU=0.0665
[15:18:29]   [Val] epoch=1  loss=1.4698  IoU=0.0879  RMSE=0.5016  MAE=0.4454
[15:18:29]   LR → 1.00e-06
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/main/latest.pth  (102.5 MB)
  [ckpt] Saved → /tmp/SPATIAL-LLAVA/checkpoints/main/best.pth  (102.5 MB)
  [ckpt] ★ New best IoU: 0.0879
[15:18:30] 
[5/5] Saving results ...
[15:18:30]   Saved metrics → /tmp/SPATIAL-LLAVA/results/main/metrics.json
[15:18:30]   Saved predictions → /tmp/SPATIAL-LLAVA/results/main/predictions.json
[15:18:30]   Saved 0 example PNGs → /tmp/SPATIAL-LLAVA/results/main/examples
[15:18:30] 
✅ Step 2 complete!
[15:18:30]   Best val IoU  : 0.0879
[15:18:30]   Checkpoint    : /tmp/SPATIAL-LLAVA/checkpoints/main/best.pth
[15:18:30]   Results       : /tmp/SPATIAL-LLAVA/results/main


# Test Flash-Attn

In [1]:
from core.model.spatial_llava import load_model
model_check, _ = load_model(use_lora=False, device='cuda')
print(model_check.backbone.model.language_model.layers[0].self_attn.__class__.__name__)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[load_model] use_lora=False  device=cuda


[SpatialLLaVA] Loading llava-hf/llava-1.5-7b-hf ...


Loading weights: 100%|██████████| 686/686 [00:08<00:00, 76.25it/s] 


[SpatialLLaVA] Freezing entire backbone (ablation mode) ...
[SpatialLLaVA] Head params: 2,230,020
LlamaAttention


In [2]:
import transformers
print(transformers.__version__)

# 看实际有哪些 attention 相关的类
for name, module in model_check.backbone.model.language_model.named_modules():
    if 'attn' in name and 'layers.0' in name:
        print(name, type(module).__name__)
        break

5.4.0
layers.0.self_attn LlamaAttention


In [3]:
# 看 LlamaAttention 的实现细节
attn = model_check.backbone.model.language_model.layers[0].self_attn
print(dir(attn))
print(hasattr(attn, '_flash_attn_uses_top_left_mask'))
print(type(attn.config) if hasattr(attn, 'config') else 'no config')

# 检查 config 里有没有 attn_implementation
print(model_check.backbone.config)

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_is_hf_initialized', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_l

In [4]:
# transformers 5.x 的新方式
from transformers.utils import is_flash_attn_2_available
print("flash_attn available:", is_flash_attn_2_available())

# 看 LlamaAttention 的 forward 源码用的是什么
import inspect
attn = model_check.backbone.model.language_model.layers[0].self_attn
print(inspect.getfile(type(attn)))

flash_attn available: True
/opt/conda/lib/python3.11/site-packages/transformers/models/llama/modeling_llama.py


In [5]:
import inspect
src = inspect.getsource(type(attn).forward)
print(src[:500])

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        attention_mask: torch.Tensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden


In [6]:
# transformers 5.x 用 SDPA 自动选择最优 attention
# 看看实际用的是什么 kernel
import torch
print(torch.backends.cuda.flash_sdp_enabled())
print(torch.backends.cuda.mem_efficient_sdp_enabled())
print(torch.backends.cuda.math_sdp_enabled())

True
True
True
